# L4b Algorithm: Breadth-First Search

Breadth-first search (BFS) explores a directed graph in layers: it processes the starting vertex, then every vertex one edge away, then every newly discovered vertex two edges away, and so on. A first-in, first-out queue preserves that discovery order.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
> * __Trace a FIFO graph traversal:__ Follow the queue, visited set, and first-visit order after each processed vertex. Distinguish vertices waiting in the queue from vertices whose outgoing edges have already been examined.
> * __Explain discovery-time marking:__ Show why marking a vertex when it enters the queue prevents two frontier vertices from enqueuing the same neighbor. Use that rule to explain why every reachable vertex is eventually processed once.
> * __Connect the invariant to an implementation:__ Map initialization, queue processing, and neighbor discovery to the final three tasks in the student traversal file. Relate those stages to the running time and storage used by the traversal.

Let's get started!

___

## The Algorithm

The figure shows the directed graph used in the L4b lab. Its sorted adjacency list begins with `1 => [2, 3]`, so vertex 2 is discovered before vertex 3 whenever both are first reached from vertex 1.

<div>
    <center>
        <img src="figs/Fig-Example-Graph.svg" width="480" alt="Six-vertex directed graph used for the breadth-first-search trace"/>
    </center>
</div>

> __BFS queue invariant__
>
> Every vertex in the queue has been discovered and marked visited, but its outgoing edges have not yet been processed. Vertices ahead of it were discovered no later, so removing from the front processes the graph in nondecreasing directed distance from the start. A vertex enters the queue at most once because discovery and marking happen together.

__Initialization:__ Given an adjacency list for a directed graph and a starting vertex $v_s$, create an empty first-visit order, mark $v_s$ visited, and place $v_s$ at the back of an empty queue.

Repeat the following steps until the queue is empty:

1. Remove the vertex at the front of the queue and append it to the first-visit order.
2. Examine that vertex's outgoing neighbors in ascending identifier order.
3. For each neighbor not already visited, mark it immediately and append it to the back of the queue.

The queue holds the boundary between discovered work and processed work. Marking on enqueue, rather than waiting until dequeue, is what prevents duplicate queue entries when several vertices point to the same neighbor.

___

## Trace the Queue

Starting at vertex 1 gives the following state transitions. The queue column records only vertices still waiting to be processed; the order column records vertices after they leave the queue.

| Step | Processed vertex | Newly enqueued | Queue after the step | Visited after the step | First-visit order |
|:--|:--:|:--|:--|:--|:--|
| Initialize | none | `1` | `[1]` | `{1}` | `[]` |
| 1 | `1` | `2, 3` | `[2, 3]` | `{1, 2, 3}` | `[1]` |
| 2 | `2` | `4` | `[3, 4]` | `{1, 2, 3, 4}` | `[1, 2]` |
| 3 | `3` | `5` | `[4, 5]` | `{1, 2, 3, 4, 5}` | `[1, 2, 3]` |
| 4 | `4` | `6` | `[5, 6]` | `{1, 2, 3, 4, 5, 6}` | `[1, 2, 3, 4]` |
| 5 | `5` | none | `[6]` | `{1, 2, 3, 4, 5, 6}` | `[1, 2, 3, 4, 5]` |
| 6 | `6` | none | `[]` | `{1, 2, 3, 4, 5, 6}` | `[1, 2, 3, 4, 5, 6]` |

At step 2, vertex 3 is already visited because it entered the queue during step 1, so the edge `2 → 3` does not enqueue it again. At step 5, the same rule ignores `5 → 4`. These are not special cases: the visited-set test is the mechanism that handles both converging edges and cycles.

The resulting sequence `[1, 2, 3, 4, 5, 6]` is a first-visit order, not a directed path. For example, vertices 3 and 4 are consecutive in the sequence even though the graph has no edge `3 → 4`.

___

## Implementation Contract and Cost

The final three TODOs in [`src/Compute.jl`](src/Compute.jl) divide the trace into implementation stages: seed the state and queue, process one queued vertex at a time, and discover previously unseen outgoing neighbors. A vector plus a moving head index provides FIFO behavior without repeatedly shifting every remaining element with [the `popfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.popfirst!).

Let $V_r$ and $E_r$ denote the vertices and directed edges reachable from the start. Once outgoing-neighbor lists are ordered, the queue loop processes each reachable vertex once and examines each reachable edge once, giving $\mathcal{O}(|V_r|+|E_r|)$ traversal work. The lab's general interface also sorts a copy of each neighbor collection to guarantee deterministic output; that normalization contributes $\sum_{v\in V_r}\mathcal{O}(d_v\log d_v)$ work, where $d_v$ is the out-degree of vertex $v$.

The visited set, queue, and result together require $\mathcal{O}(|V_r|)$ storage. The algorithm terminates because every loop iteration processes one queued vertex and no vertex can enter the queue more than once.

___

## Summary

Breadth-first search turns discovery order into processing order with a FIFO queue, producing directed-distance layers from the selected start.

> __Key Takeaways:__
>
> * __The queue separates discovered from processed vertices:__ Enqueued vertices are known and marked but still have outgoing edges to examine. Removing from the front preserves the order in which the frontier was discovered.
> * __Marking on enqueue prevents duplicate work:__ Once a vertex enters the queue, every later incoming edge recognizes it as visited. That rule keeps converging edges and cycles from creating repeated queue entries.
> * __Layer order is not a path:__ Breadth-first search records when each reachable vertex is first processed. Consecutive entries can come from different branches and need not share an edge, even though their positions respect directed distance from the start.

Return to [the L4b lab](CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) and use this queue invariant to complete TODOs 4 through 6 in [`src/Compute.jl`](src/Compute.jl).

___